In [2]:
import argparse
import logging
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path
from typing import Iterable

In [3]:
LOGGER = logging.getLogger("kotak_trader.candle_creator")

DEFAULT_START_TIME = "09:15"
DEFAULT_END_TIME = "15:30"
DEFAULT_CANDLE_MINUTES = 5
DEFAULT_DB_PATH = Path("data/market_data.db")

In [5]:
db_path = Path("C:/Projects/Kotak Trader/data/market_data.db")
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

In [48]:
get_ohlcv = """
WITH
-- Step 1: Get the base integer timestamp
raw_data as ( select       mt.instrument_token,
        -- Convert DD/MM/YYYY HH:MM:SS
        -- to YYYY-MM-DD HH:MM:SS
        substr(mt.last_trade_timestamp, 7, 4) || '-' ||
        substr(mt.last_trade_timestamp, 4, 2) || '-' ||
        substr(mt.last_trade_timestamp, 1, 2) || ' ' ||
        substr(mt.last_trade_timestamp, 12, 8) AS tick_time,
        last_trade_timestamp,
        mt.ltp,
        mt.volume
        from market_ticks as mt
        where last_trade_timestamp is not null
),
raw_epochs AS (
    SELECT 
        *,
        CAST(strftime('%s', tick_time) AS INTEGER) AS epoch_seconds
    FROM raw_data
),

-- Step 2: Calculate the correct candle bucket (assuming 5 minute candles as an example)
-- Replace the hardcoded 5 with your ? parameter or f-string variable
candle_buckets AS (
    SELECT 
        *,
        epoch_seconds - (epoch_seconds % (5 * 60)) AS candle_epoch
    FROM raw_epochs
),

-- Step 3: Now that candle_epoch exists, we can use it in datetime() and window functions
candle_groups AS (
    SELECT 
        *,
        datetime(candle_epoch, 'unixepoch') AS candle_time,
        ROW_NUMBER() OVER (
            PARTITION BY instrument_token, candle_epoch 
            ORDER BY last_trade_timestamp ASC
        ) AS rn_open,
        ROW_NUMBER() OVER (
            PARTITION BY instrument_token, candle_epoch 
            ORDER BY last_trade_timestamp DESC
        ) AS rn_close
    FROM candle_buckets
)

-- Final Step: Aggregate (Notice NO comma before SELECT)
SELECT 
    instrument_token,
    candle_time,
    MAX(CASE WHEN rn_open = 1 THEN ltp END) AS open,
    MAX(ltp) AS high,
    MIN(ltp) AS low,
    MAX(CASE WHEN rn_close = 1 THEN ltp END) AS close,
    SUM(volume) AS volume
FROM candle_groups
GROUP BY 
    instrument_token, 
    candle_time
ORDER BY 
    instrument_token, 
    candle_time;
"""

In [10]:
get_symbols = '''
select * from instruments where instrument_token in (select distinct instrument_token from market_ticks);
'''

data = conn.execute(get_symbols).fetchall()

In [16]:
for row in data:
    print(row['trading_symbol'])

NIFTY26OCTFUT
NIFTY26AUGFUT
NIFTY26AUG24400CE
NIFTY26AUG24400PE
NIFTY26SEPFUT


In [49]:
import pandas as pd

ohlcv = pd.read_sql_query(get_ohlcv, conn)

In [51]:
ohlcv.head(10)

,instrument_token,candle_time,open,high,low,close,volume
0,48704,2026-08-14 15:35:00,24726.1,24726.1,24726.1,24726.1,153140
1,48704,2026-08-17 11:10:00,24575.5,24575.5,24575.5,NaN,320190
2,48704,2026-08-17 11:15:00,24563.0,24574.0,24563.0,24574.0,520455
3,48704,2026-08-17 11:20:00,24582.5,24582.5,24582.5,NaN,678990
4,48704,2026-08-17 11:25:00,24576.0,24580.0,24570.0,NaN,976690
5,48704,2026-08-17 11:30:00,24583.0,24583.2,24580.0,NaN,498810
6,48704,2026-08-17 11:35:00,24590.0,24593.3,24590.0,NaN,720395
7,48704,2026-08-17 11:40:00,NaN,24591.2,24591.2,24591.2,145665
8,48704,2026-08-17 11:45:00,NaN,24595.7,24590.0,NaN,659685
9,48704,2026-08-17 11:50:00,NaN,NaN,NaN,NaN,73710


In [58]:
pd.read_sql_query("select * from market_ticks order by tick_id desc limit 50", conn)

,tick_id,message_id,received_at_ns,feed_timestamp,last_trade_timestamp,instrument_token,exchange_segment,ltp,change,change_pct,volume,open_interest,turnover,bid_price,ask_price,bid_quantity,ask_quantity,total_buy_quantity,total_sell_quantity,average_price
0,274535,80834,1787324329477262500,None,None,48704,nse_fo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,274534,80833,1787324328417268700,21/08/2026 16:53:16,2026-08-21T15:39:56,48704,nse_fo,24516.10,-2.70,-0.01,107185.0,693030.0,2.627388e+09,24515.00,24520.00,65.0,130.0,26455.0,32890.0,24512.65
2,274533,80833,1787324328417268700,21/08/2026 17:05:18,2026-08-21T15:39:59,61726,nse_fo,148.70,-4.30,-2.81,65975260.0,3521960.0,9.828994e+09,148.60,149.15,520.0,1690.0,145210.0,208195.0,148.98
3,274532,80833,1787324328417268700,21/08/2026 17:05:18,2026-08-21T15:39:59,61720,nse_fo,33.00,-14.35,-30.31,127463570.0,9774440.0,4.792630e+09,32.80,33.20,975.0,65.0,290160.0,961870.0,37.60
4,274531,80833,1787324328417268700,21/08/2026 16:54:04,2026-08-21T15:39:59,58072,nse_fo,24288.00,-5.00,-0.02,2402920.0,11113960.0,5.837010e+10,24281.30,24289.00,195.0,195.0,494715.0,462085.0,24291.32
5,274530,80833,1787324328417268700,21/08/2026 16:54:36,2026-08-21T15:39:56,68407,nse_fo,24394.90,2.30,0.01,1948180.0,6720415.0,4.752266e+10,24388.00,24394.90,1430.0,520.0,74100.0,106275.0,24393.36
6,274529,80832,1787324175170475800,21/08/2026 17:05:18,None,61726,nse_fo,148.70,-4.30,-2.81,65975260.0,3521960.0,9.828994e+09,148.60,149.15,520.0,1690.0,145210.0,208195.0,148.98
7,274528,80832,1787324175170475800,21/08/2026 16:53:16,None,48704,nse_fo,24516.10,-2.70,-0.01,107185.0,693030.0,2.627388e+09,24515.00,24520.00,65.0,130.0,26455.0,32890.0,24512.65
8,274527,80832,1787324175170475800,21/08/2026 17:05:18,None,61720,nse_fo,33.00,-14.35,-30.31,127463570.0,9774440.0,4.792630e+09,32.80,33.20,975.0,65.0,290160.0,961870.0,37.60
9,274526,80832,1787324175170475800,21/08/2026 16:54:04,None,58072,nse_fo,24288.00,-5.00,-0.02,2402920.0,11113960.0,5.837010e+10,24281.30,24289.00,195.0,195.0,494715.0,462085.0,24291.32
